# Agent Comparison

This notebook compares the current Hanabi baselines:

- `RandomAgent`
- `BasicHeuristicAgent`
- `ConventionHeuristicAgent`
- `TempoHeuristicAgent`

It is meant to serve as the first lightweight record of agent metrics in the repository.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"

try:
    import hanabi_ai  # noqa: F401
    print("Using installed hanabi_ai package.")
except ModuleNotFoundError:
    if str(SRC_PATH) not in sys.path:
        sys.path.insert(0, str(SRC_PATH))
    print("hanabi_ai not installed; using src/ fallback.")

print(f"Project root: {PROJECT_ROOT}")

Using installed hanabi_ai package.
Project root: c:\Users\marce\Documents\GitHub\hanabi-ai


In [2]:
from hanabi_ai.agents.random import RandomAgent
from hanabi_ai.agents.heuristic.basic import BasicHeuristicAgent
from hanabi_ai.agents.heuristic.convention import ConventionHeuristicAgent
from hanabi_ai.agents.heuristic.tempo import TempoHeuristicAgent
from hanabi_ai.training.self_play import evaluate_self_play


In [3]:
# Edit these values when you want to record a new comparison run.
PLAYER_COUNT = 2
GAME_COUNT = 200
SEED_BASE = 0
AGENT_SEED_BASE = 1000

In [4]:
basic_evaluation = evaluate_self_play(
    lambda player_id, game_index: BasicHeuristicAgent(),
    player_count=PLAYER_COUNT,
    game_count=GAME_COUNT,
    seed_base=SEED_BASE,
)

convention_evaluation = evaluate_self_play(
    lambda player_id, game_index: ConventionHeuristicAgent(),
    player_count=PLAYER_COUNT,
    game_count=GAME_COUNT,
    seed_base=SEED_BASE,
)

tempo_evaluation = evaluate_self_play(
    lambda player_id, game_index: TempoHeuristicAgent(),
    player_count=PLAYER_COUNT,
    game_count=GAME_COUNT,
    seed_base=SEED_BASE,
)

random_evaluation = evaluate_self_play(
    lambda player_id, game_index: RandomAgent(
        seed=AGENT_SEED_BASE + (game_index * PLAYER_COUNT) + player_id
    ),
    player_count=PLAYER_COUNT,
    game_count=GAME_COUNT,
    seed_base=SEED_BASE,
)


In [5]:
def evaluation_to_row(name, evaluation):
    return {
        "agent": name,
        "games": evaluation.game_count,
        "players": evaluation.player_count,
        "avg_score": round(evaluation.average_score, 3),
        "median_score": round(evaluation.median_score, 3),
        "min_score": evaluation.min_score,
        "max_score": evaluation.max_score,
        "avg_turns": round(evaluation.average_turn_count, 3),
        "avg_hints": round(evaluation.average_hint_tokens, 3),
        "avg_strikes": round(evaluation.average_strike_tokens, 3),
        "avg_completed_stacks": round(evaluation.average_completed_stacks, 3),
        "avg_play_actions": round(evaluation.average_play_actions, 3),
        "avg_successful_plays": round(evaluation.average_successful_plays, 3),
        "avg_failed_plays": round(evaluation.average_failed_plays, 3),
        "avg_discards": round(evaluation.average_discards, 3),
        "avg_hints_given": round(evaluation.average_hints_given, 3),
        "successful_play_rate": round(evaluation.successful_play_rate, 4),
        "win_rate": round(evaluation.win_rate, 4),
        "loss_rate": round(evaluation.loss_rate, 4),
        "score>=10": round(evaluation.score_at_least_10_rate, 4),
        "score>=15": round(evaluation.score_at_least_15_rate, 4),
    }


rows = [
    evaluation_to_row("RandomAgent", random_evaluation),
    evaluation_to_row("BasicHeuristicAgent", basic_evaluation),
    evaluation_to_row("ConventionHeuristicAgent", convention_evaluation),
    evaluation_to_row("TempoHeuristicAgent", tempo_evaluation),
]


def safe_ratio(numerator, denominator):
    return round(numerator / denominator, 4) if denominator else None


derived_rows = [
    {
        "agent": row["agent"],
        "score_per_hint": safe_ratio(row["avg_score"], row["avg_hints_given"]),
        "score_per_play": safe_ratio(row["avg_score"], row["avg_play_actions"]),
        "plays_per_hint": safe_ratio(row["avg_play_actions"], row["avg_hints_given"]),
        "discards_per_score": safe_ratio(row["avg_discards"], row["avg_score"]),
        "turns_per_score": safe_ratio(row["avg_turns"], row["avg_score"]),
    }
    for row in rows
]

In [6]:
rows[0]

{'agent': 'RandomAgent',
 'games': 200,
 'players': 2,
 'avg_score': 1.415,
 'median_score': 1.0,
 'min_score': 0,
 'max_score': 8,
 'avg_turns': 13.93,
 'avg_hints': 4.855,
 'avg_strikes': 3.0,
 'avg_completed_stacks': 0.0,
 'avg_play_actions': 4.415,
 'avg_successful_plays': 1.415,
 'avg_failed_plays': 3.0,
 'avg_discards': 3.185,
 'avg_hints_given': 6.33,
 'successful_play_rate': 0.3205,
 'win_rate': 0.0,
 'loss_rate': 1.0,
 'score>=10': 0.0,
 'score>=15': 0.0}

In [7]:
rows[1]

{'agent': 'BasicHeuristicAgent',
 'games': 200,
 'players': 2,
 'avg_score': 6.97,
 'median_score': 7.0,
 'min_score': 2,
 'max_score': 12,
 'avg_turns': 81.51,
 'avg_hints': 0.32,
 'avg_strikes': 0.0,
 'avg_completed_stacks': 0.09,
 'avg_play_actions': 6.97,
 'avg_successful_plays': 6.97,
 'avg_failed_plays': 0.0,
 'avg_discards': 33.385,
 'avg_hints_given': 41.155,
 'successful_play_rate': 1.0,
 'win_rate': 0.0,
 'loss_rate': 0.0,
 'score>=10': 0.155,
 'score>=15': 0.0}

In [8]:
rows[2]

{'agent': 'ConventionHeuristicAgent',
 'games': 200,
 'players': 2,
 'avg_score': 8.61,
 'median_score': 8.0,
 'min_score': 3,
 'max_score': 17,
 'avg_turns': 79.935,
 'avg_hints': 0.29,
 'avg_strikes': 0.0,
 'avg_completed_stacks': 0.155,
 'avg_play_actions': 8.61,
 'avg_successful_plays': 8.61,
 'avg_failed_plays': 0.0,
 'avg_discards': 31.73,
 'avg_hints_given': 39.595,
 'successful_play_rate': 1.0,
 'win_rate': 0.0,
 'loss_rate': 0.0,
 'score>=10': 0.36,
 'score>=15': 0.035}

In [9]:
headers = list(rows[0].keys())
widths = {
    header: max(len(header), max(len(str(row[header])) for row in rows))
    for header in headers
}

header_line = " | ".join(header.ljust(widths[header]) for header in headers)
separator_line = "-+-".join("-" * widths[header] for header in headers)

print(header_line)
print(separator_line)
for row in rows:
    print(" | ".join(str(row[header]).ljust(widths[header]) for header in headers))

print()
print("Derived efficiency metrics")

derived_headers = list(derived_rows[0].keys())
derived_widths = {
    header: max(len(header), max(len(str(row[header])) for row in derived_rows))
    for header in derived_headers
}

derived_header_line = " | ".join(
    header.ljust(derived_widths[header]) for header in derived_headers
)
derived_separator_line = "-+-".join(
    "-" * derived_widths[header] for header in derived_headers
)

print(derived_header_line)
print(derived_separator_line)
for row in derived_rows:
    print(" | ".join(str(row[header]).ljust(derived_widths[header]) for header in derived_headers))

agent                      | games | players | avg_score | median_score | min_score | max_score | avg_turns | avg_hints | avg_strikes | avg_completed_stacks | avg_play_actions | avg_successful_plays | avg_failed_plays | avg_discards | avg_hints_given | successful_play_rate | win_rate | loss_rate | score>=10 | score>=15
---------------------------+-------+---------+-----------+--------------+-----------+-----------+-----------+-----------+-------------+----------------------+------------------+----------------------+------------------+--------------+-----------------+----------------------+----------+-----------+-----------+----------
RandomAgent                | 200   | 2       | 1.415     | 1.0          | 0         | 8         | 13.93     | 4.855     | 3.0         | 0.0                  | 4.415            | 1.415                | 3.0              | 3.185        | 6.33            | 0.3205               | 0.0      | 1.0       | 0.0       | 0.0      
BasicHeuristicAgent        | 200   | 

In [10]:
for name, evaluation in [
    ("RandomAgent", random_evaluation),
    ("BasicHeuristicAgent", basic_evaluation),
    ("ConventionHeuristicAgent", convention_evaluation),
    ("TempoHeuristicAgent", tempo_evaluation),
]:
    print(name)
    print("  score_distribution:", evaluation.score_distribution)
    print("  average_play_actions:", round(evaluation.average_play_actions, 3))
    print("  average_successful_plays:", round(evaluation.average_successful_plays, 3))
    print("  average_failed_plays:", round(evaluation.average_failed_plays, 3))
    print("  average_discards:", round(evaluation.average_discards, 3))
    print("  average_hints_given:", round(evaluation.average_hints_given, 3))
    print("  successful_play_rate:", round(evaluation.successful_play_rate, 4))
    print()

RandomAgent
  score_distribution: ((0, 57), (1, 66), (2, 39), (3, 25), (4, 6), (5, 4), (6, 2), (8, 1))
  average_play_actions: 4.415
  average_successful_plays: 1.415
  average_failed_plays: 3.0
  average_discards: 3.185
  average_hints_given: 6.33
  successful_play_rate: 0.3205

BasicHeuristicAgent
  score_distribution: ((2, 4), (3, 17), (4, 17), (5, 20), (6, 23), (7, 30), (8, 29), (9, 29), (10, 17), (11, 12), (12, 2))
  average_play_actions: 6.97
  average_successful_plays: 6.97
  average_failed_plays: 0.0
  average_discards: 33.385
  average_hints_given: 41.155
  successful_play_rate: 1.0

ConventionHeuristicAgent
  score_distribution: ((3, 4), (4, 11), (5, 10), (6, 20), (7, 18), (8, 39), (9, 26), (10, 27), (11, 19), (12, 12), (13, 6), (14, 1), (15, 4), (16, 2), (17, 1))
  average_play_actions: 8.61
  average_successful_plays: 8.61
  average_failed_plays: 0.0
  average_discards: 31.73
  average_hints_given: 39.595
  successful_play_rate: 1.0

